# HG4052 · Week 8 Practical
## The CTC collapse, then Whisper on Singlish

**No installs: everything used here is preinstalled on Colab. Headphones on.**

By the end you will have:
- ✅ the CTC collapse, finished by you, decoding the lecture's eight frames to "no"
- ✅ two Whisper models (the original and a Singlish fine-tune) transcribing a clean clip and your own voice
- ✅ a small WER table: two models, two clips
- ✅ a record of what both models write for five seconds of silence

**How this notebook works.** Same as every week: click a cell, press **Shift + Enter**, and read the output underneath. There is one **✏️ TODO** in code, a single line. Written TODOs are answered by double-clicking the cell and typing.

**Short on time?** Do Parts 1 to 4 in order; each is a few cells. The two take-home cells at the end are optional.

---
## 0 · Setup

In [ ]:
import os, re, urllib.request, warnings
import numpy as np
import librosa
from IPython.display import Audio, display

warnings.filterwarnings("ignore")

os.makedirs("data", exist_ok=True)
for name in ["clean_read.wav", "fallback_singlish.wav"]:
    path = f"data/{name}"
    if not os.path.exists(path):
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/chenchenzi/hg4052-materials/main/week08/" + name, path)
        except Exception as e:
            print("download failed:", e)
    ok = os.path.exists(path) and os.path.getsize(path) > 10_000
    print(("✅" if ok else "❌"), f"{name:<22}", (f"{os.path.getsize(path):,} bytes" if ok else "missing (also on NTULearn: download it and upload into data/)"))

---
## 1 · The CTC collapse

A CTC model writes one label per frame: a letter, or the blank ∅ ("no new letter at this frame"). The collapse turns the frame labels into text, in two steps:

1. **merge repeats**: `n n ∅ o o o ∅ ∅` becomes `n ∅ o ∅`
2. **delete blanks**: `n ∅ o ∅` becomes `no`

Step 1 is written for you. **✏️ TODO (one line):** write step 2. If you leave it, a finished copy takes over so the rest of the notebook runs.

In [ ]:
BLANK = "∅"

def collapse(labels):
    merged = []
    for s in labels:                          # step 1: merge repeats
        if not merged or s != merged[-1]:     # keep a label only if it differs from the one before
            merged.append(s)
    # ✏️ TODO step 2: keep every label in merged that is not BLANK.
    # Shape of the answer:   letters = [s for s in merged if ...]
    letters = ...
    return "".join(letters)

def collapse_reference(labels):
    merged = []
    for s in labels:
        if not merged or s != merged[-1]:
            merged.append(s)
    return "".join(s for s in merged if s != BLANK)

try:
    TODO_DONE = collapse(list("nn∅ooo∅∅")) == "no"
except TypeError:
    TODO_DONE = False
COLLAPSE = collapse if TODO_DONE else collapse_reference
print("✅ your collapse works" if TODO_DONE else "⬆ TODO not finished: using collapse_reference for now")

**The lecture's eight frames.** Each row is one frame: the network's probabilities for n, o and ∅ (illustrative numbers, the same as on the slides). Greedy decoding takes the most probable label in each frame, then collapses.

In [ ]:
LABELS = ["n", "o", BLANK]
P = np.array([[0.7, 0.1, 0.2],
              [0.6, 0.2, 0.2],
              [0.2, 0.2, 0.6],
              [0.1, 0.7, 0.2],
              [0.1, 0.8, 0.1],
              [0.2, 0.6, 0.2],
              [0.1, 0.2, 0.7],
              [0.1, 0.1, 0.8]])

best = P.argmax(axis=1)                       # the most probable label in each frame
path = [LABELS[i] for i in best]
print("frame:      ", "  ".join(f"x{t+1}" for t in range(8)))
print("best label: ", "   ".join(path))
print("collapsed:  ", repr(COLLAPSE(path)))
print(f"probability of this one path: {np.prod(P.max(axis=1)):.3f}")

**Predict, then run.** What does each frame string below collapse to? Write your guesses first (double-click), then run the cell.

> `b o o k`: **...**  `b o ∅ o k`: **...**  `∅ ∅ ∅ ∅`: **...**

In [ ]:
for s in ["b o o k", "b o ∅ o k", "∅ ∅ ∅ ∅"]:
    print(f"{s:<12} →  {COLLAPSE(s.split())!r}")

**✏️ TODO (in words, double-click).** Why does CTC need the blank to write a word like *book*?

> **...**

---
## 2 · Whisper listens

Two models of the same size (whisper-small, 244 million parameters):
- **A:** `openai/whisper-small`, the original
- **B:** `jensenlwt/whisper-small-singlish-122k`, the same model fine-tuned on Singapore English (the National Speech Corpus)

The next cell downloads both (about 1 GB each; one or two minutes). A GPU is optional: Runtime → Change runtime type → T4 GPU makes it faster.

In [ ]:
import torch
from transformers import pipeline
from transformers.utils import logging
logging.set_verbosity_error()

DEVICE = 0 if torch.cuda.is_available() else -1
MODELS = {"A (original)": "openai/whisper-small",
          "B (Singlish fine-tune)": "jensenlwt/whisper-small-singlish-122k"}
asr = {}
for label, name in MODELS.items():
    try:
        asr[label] = pipeline("automatic-speech-recognition", model=name, device=DEVICE)
        print("✅", label, "loaded:", name)
    except Exception as e:
        print("❌", label, "did not load:", type(e).__name__)

# Stored transcripts from a test run, used only if a model did not load (provided clips only).
STORED = {
    ("A (original)", "clean_read"): " We went to the market on Saturday morning and bought some fruit for the week.",
    ("A (original)", "fallback_singlish"): " Ken Law, we go back in at the copy team after this.",
    ("A (original)", "silence"): " you",
    ("B (Singlish fine-tune)", "clean_read"): " We went to the Market on Saturday morning and bought some Fruit for the weak",
    ("B (Singlish fine-tune)", "fallback_singlish"): "Kenla we go back in at the Copyteam after this",
    ("B (Singlish fine-tune)", "silence"): " you",
}

def transcribe(label, audio, clip_name):
    if label in asr:
        out = asr[label]({"raw": audio, "sampling_rate": 16000},
                         generate_kwargs={"language": "english", "task": "transcribe"})
        return out["text"]
    return STORED.get((label, clip_name), "(model not loaded)")

**Clip 1: clean read speech.** A synthetic voice reading one sentence. Listen, then transcribe it with both models.

In [ ]:
CLEAN_REF = "We went to the market on Saturday morning and bought some fruit for the week."
clean, _ = librosa.load("data/clean_read.wav", sr=16000)
display(Audio(clean, rate=16000))

hyp = {}
for label in MODELS:
    hyp[(label, "clean_read")] = transcribe(label, clean, "clean_read")
    print(f"{label:<24} {hyp[(label, 'clean_read')]!r}")

**Clip 2: your own voice.** Record yourself saying this sentence (or any Singapore English sentence you like):

> *Can lah, we go makan at the kopitiam after this.*

Use your laptop's voice recorder, your phone, or Praat as in Week 3. WAV, MP3 and M4A all work.

To upload: delete the leading `#` from the five commented lines and run the cell. A file dialog opens. Then change `MY_REFERENCE` to exactly what you said. No recording? Run the cell as it is: it uses a fallback clip, a **synthetic American voice** reading the sentence. The fallback tests the Singlish words, not a Singapore accent.

In [ ]:
MY_CLIP = "data/fallback_singlish.wav"
MY_REFERENCE = "Can lah, we go makan at the kopitiam after this."      # ✏️ change this to exactly what you said

# from google.colab import files
# import shutil
# uploaded = files.upload()
# for name in uploaded:
#     shutil.move(name, "data/" + name); MY_CLIP = "data/" + name

mine, _ = librosa.load(MY_CLIP, sr=16000)
MINE_NAME = "fallback_singlish" if MY_CLIP.endswith("fallback_singlish.wav") else "my_voice"
print("Working file:", MY_CLIP, f"({len(mine) / 16000:.1f} s)")
display(Audio(mine, rate=16000))

for label in MODELS:
    hyp[(label, "my_clip")] = transcribe(label, mine, MINE_NAME)
    print(f"{label:<24} {hyp[(label, 'my_clip')]!r}")

---
## 3 · Score it: normalize, then WER

Model B writes no capitals and no full stop; model A does. Should that count as an error? Not here. A **normalizer** cleans both sides the same way before scoring: lowercase, strip punctuation. Then WER is Week 5's word edit distance divided by the number of reference words.

In [ ]:
def normalize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s']", " ", text)        # punctuation becomes a space (apostrophes stay)
    return " ".join(text.split())

def wer(reference, hypothesis):
    r, h = normalize(reference).split(), normalize(hypothesis).split()
    D = np.zeros((len(r) + 1, len(h) + 1), dtype=int)
    D[:, 0] = np.arange(len(r) + 1)
    D[0, :] = np.arange(len(h) + 1)
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            same = 0 if r[i - 1] == h[j - 1] else 1
            D[i, j] = min(D[i - 1, j] + 1, D[i, j - 1] + 1, D[i - 1, j - 1] + same)
    return D[len(r), len(h)] / len(r)

REFS = {"clean_read": CLEAN_REF, "my_clip": MY_REFERENCE}
print(f"{'clip':<12} {'model':<24} {'WER':>6}   normalized hypothesis")
for clip, ref in REFS.items():
    print(f"{clip:<12} {'(reference)':<24} {'':>6}   {normalize(ref)}")
    for label in MODELS:
        h = hyp[(label, clip)]
        print(f"{clip:<12} {label:<24} {wer(ref, h):6.0%}   {normalize(h)}")
    print()

**✏️ TODO (in words, double-click).** Read the errors phonetically. Pick two wrong words. For each: what was said, what was written, and what in the sound could have led the model there? Did the fine-tune fix the particle (*lah*) or the borrowed words (*makan*, *kopitiam*)?

> **...**

---
## 4 · The silence test

Five seconds of digital silence: every sample is 0. **Predict first:** what will each model write?

In [ ]:
silence = np.zeros(5 * 16000, dtype=np.float32)
for label in MODELS:
    print(f"{label:<24} {transcribe(label, silence, 'silence')!r}")

**✏️ TODO (in words, two lines, double-click).** What did each model write? Whisper was trained on 680,000 hours of audio with transcripts found on the web, many of them subtitles. Why might such a model write words for silence?

> **...**

---
## Take-home (optional)

**(a) Greedy vs the sum.** Two frames, one letter *a*. On both frames P(a) = 0.4 and P(∅) = 0.6 (illustrative numbers). The cell lists all four paths, collapses each with your function, and adds up the probability of each transcript. Why does greedy decoding output nothing, when "a" is more probable?

In [ ]:
from itertools import product
p = {"a": 0.4, BLANK: 0.6}
totals = {}
for path2 in product(["a", BLANK], repeat=2):
    prob = p[path2[0]] * p[path2[1]]
    text = COLLAPSE(list(path2))
    totals[text] = totals.get(text, 0) + prob
    print(" ".join(path2), " →", repr(text), f" P = {prob:.2f}")
print("totals:", {k: round(v, 2) for k, v in totals.items()})

**(b) For Assignment 1.** Steps 2 to 4 of the A1 brief are Parts 2 and 3 of this notebook, run over your own test set. The cell below is a start: put your clips in `data/a1/`, type one reference per file, and it writes one row per clip with both hypotheses and both WERs. Before you upload any corpus audio to Colab, check the data rules in the A1 brief.

In [ ]:
import csv
A1_REFS = {
    # "clip001.wav": "the reference transcript of clip001",
}
rows = []
for fname, ref in A1_REFS.items():
    audio, _ = librosa.load(os.path.join("data/a1", fname), sr=16000)
    row = {"file": fname, "reference": ref}
    for label in MODELS:
        row[label] = transcribe(label, audio, fname)
        row["WER " + label] = round(wer(ref, row[label]), 3)
    rows.append(row)
if rows:
    with open("a1_results.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader(); w.writerows(rows)
    print(f"✅ {len(rows)} rows written to a1_results.csv")
else:
    print("A1_REFS is empty: add your clips and references first")

---
## ✅ Done looks like

- a collapse function you finished, decoding `n n ∅ o o o ∅ ∅` to "no"
- a reason why *book* needs a blank between its two o's
- two transcripts per clip, a WER for each, and two errors read phonetically
- what each model wrote for silence, and a two-line diagnosis

**What you built.** The two halves of today's lecture: the collapse that turns CTC's frame labels into text, and an end-to-end model scored the way Assignment 1 scores it. Next week: scoring honestly, measuring bias, and forced alignment.